## Komplettprogramm Bereiniung und Laden in Datenbank

In [ ]:
# -----------------------------

# Jupyter Notebook: CSV nach PostgreSQL importieren

# -----------------------------

# 1️⃣ Module importieren

import pandas as pd
import psycopg2

# -----------------------------

# 2️⃣ CSV vorbereiten

# -----------------------------

# CSV einlesen

df = pd.read_csv("listings_Berlin.csv")

# Alle Spalten als string (Text)

df = df.astype(str)

# price: nur Zahlen extrahieren, sonst leeren String

df['price'] = df['price'].str.extract('(\d+)')[0]

# NaN in leeren String umwandeln

df = df.fillna('')

# Optional: in neue CSV speichern

df.to_csv("listings_clean.csv", index=False)

# -----------------------------

# 3️⃣ Verbindung zu PostgreSQL

# -----------------------------

conn = psycopg2.connect(
host="localhost",        # oder dein Host
database="berlin_db",
user="postgres",         # dein Benutzername
password="DEIN_PASSWORT" # dein Passwort
)

cur = conn.cursor()

# -----------------------------

# 4️⃣ Tabelle erstellen (alle Spalten TEXT)

# -----------------------------

create_table_sql = """
DROP TABLE IF EXISTS listings_berlin;

CREATE TABLE listings_berlin (
id TEXT,
name TEXT,
host_id TEXT,
host_name TEXT,
neighbourhood_group TEXT,
neighbourhood TEXT,
latitude TEXT,
longitude TEXT,
room_type TEXT,
price TEXT,
minimum_nights TEXT,
number_of_reviews TEXT,
last_review TEXT,
reviews_per_month TEXT,
calculated_host_listings_count TEXT,
availability_365 TEXT,
number_of_reviews_ltm TEXT,
license TEXT
);
"""

cur.execute(create_table_sql)
conn.commit()

# -----------------------------

# 5️⃣ Daten aus CSV importieren

# -----------------------------

# In PostgreSQL gibt es COPY FROM für CSV

with open("listings_clean.csv", "r", encoding="utf-8") as f:
# Header wird übersprungen
next(f)
cur.copy_from(f, 'listings_berlin', sep=',', null='', columns=(
'id','name','host_id','host_name','neighbourhood_group',
'neighbourhood','latitude','longitude','room_type','price',
'minimum_nights','number_of_reviews','last_review',
'reviews_per_month','calculated_host_listings_count',
'availability_365','number_of_reviews_ltm','license'
))

conn.commit()

# -----------------------------

# 6️⃣ price-Spalte zu INTEGER konvertieren

# -----------------------------

cur.execute("""
ALTER TABLE listings_berlin
ALTER COLUMN price TYPE INTEGER USING price::INTEGER;
""")
conn.commit()

# -----------------------------

# 7️⃣ Erste Abfragen testen

# -----------------------------

# Anzahl Zeilen

cur.execute("SELECT COUNT(*) FROM listings_berlin;")
print("Anzahl Zeilen:", cur.fetchone()[0])

# Head der Tabelle

cur.execute("SELECT * FROM listings_berlin LIMIT 5;")
for row in cur.fetchall():
print(row)

# Zeilen mit price < 50

cur.execute("SELECT * FROM listings_berlin WHERE price < 50 LIMIT 10;")
for row in cur.fetchall():
print(row)

# -----------------------------

# Verbindung schließen

# -----------------------------

cur.close()
conn.close()
